## Qwen3-VL-8B Full Train: 24-class + pairwise final stage

Fold 0에서 최고 성능을 만든 unweighted 24-class + pairwise 레시피를 Train 전체 9,535개에 적용합니다. Pairwise 학습 전 단계인 `fold0_cls24_v1`의 LoRA와 classifier에서 시작해 기존 Fold 모델에 pairwise 학습을 중복 적용하지 않습니다.

원본 9,535개와 Answer 분포를 보존한 입력 순서 증강 9,535개를 합쳐 19,070개로 1 epoch 학습합니다. 0.5 epoch와 1 epoch 모델을 모두 별도 저장합니다. Test 데이터와 수동 라벨은 열지 않으며 이 노트북에서는 Validation 점수를 계산하지 않습니다.

### 1. Drive·GPU·학습 시작 모델 확인

Pairwise 학습 직전의 `fold0_cls24_v1/final_adapter`를 확인하고 새 결과는 `full_pair_v1`에 저장합니다. Fold 0 모델과 기존 모델은 수정하지 않습니다.

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다.'
assert torch.cuda.is_bf16_supported(), 'BF16을 지원하는 GPU가 필요합니다.'
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/SNU')
DATA_DIR = PROJECT_DIR / 'data'
TRAIN_CSV = DATA_DIR / 'train.csv'
TRAIN_IMAGE_DIR = DATA_DIR / 'train'
MODEL_NAME = 'Qwen/Qwen3-VL-8B-Instruct'
SOURCE_RUN_NAME = 'fold0_cls24_v1'
SOURCE_DIR = PROJECT_DIR / 'outputs' / 'qwen3_vl_8b_lora' / SOURCE_RUN_NAME / 'final_adapter'
SOURCE_CLASSIFIER_PATH = SOURCE_DIR / 'permutation_classifier.safetensors'
SOURCE_MAPPING_PATH = SOURCE_DIR / 'class_mapping.json'
RUN_NAME = 'full_pair_v1'
RUN_DIR = PROJECT_DIR / 'outputs' / 'qwen3_vl_8b_lora' / RUN_NAME
HALF_ADAPTER_DIR = RUN_DIR / 'half_epoch_adapter'
HALF_CLASSIFIER_PATH = HALF_ADAPTER_DIR / 'permutation_classifier.safetensors'
HALF_MAPPING_PATH = HALF_ADAPTER_DIR / 'class_mapping.json'
FINAL_ADAPTER_DIR = RUN_DIR / 'final_adapter'
CLASSIFIER_PATH = FINAL_ADAPTER_DIR / 'permutation_classifier.safetensors'
CLASS_MAPPING_PATH = FINAL_ADAPTER_DIR / 'class_mapping.json'

for path in (TRAIN_CSV, TRAIN_IMAGE_DIR, SOURCE_DIR, SOURCE_CLASSIFIER_PATH, SOURCE_MAPPING_PATH):
    assert path.exists(), f'필수 경로가 없습니다: {path}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Pairwise 학습 시작 모델: {SOURCE_DIR}')
print(f'새 출력 폴더: {RUN_DIR}')

### 2. 패키지 설치

이전 24-class 노트북과 같은 버전을 사용합니다. PyTorch와 CUDA는 교체하지 않습니다.

In [ ]:
%pip install -q transformers==5.13.0 accelerate==1.14.0 peft==0.19.1 bitsandbytes qwen-vl-utils==0.0.14 ipywidgets
%pip uninstall -y torchao

### 3. Train 전체와 분포 보존 증강 구성

Fold를 적용하지 않고 공식 Train 9,535개를 모두 사용합니다. 이전 최고 실험과 같은 방식으로 각 원본에 입력 순서 증강을 하나씩 만들어 총 19,070개를 구성합니다. 증강 전후 24개 Answer 분포는 동일합니다.

In [ ]:
import ast
import hashlib
import json
import random
import re
import time
from dataclasses import dataclass
from itertools import permutations

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def parse_answer(value):
    answer = ast.literal_eval(value) if isinstance(value, str) else list(value)
    assert isinstance(answer, list) and len(answer) == 4 and sorted(answer) == [1, 2, 3, 4]
    return [int(number) for number in answer]

train_original = pd.read_csv(TRAIN_CSV, dtype=str, encoding='utf-8-sig')
expected_columns = [
    'Id', 'Input_1', 'Input_2', 'Input_3', 'Input_4',
    'Sentence', 'Answer', 'No_ordering',
]
assert train_original.columns.tolist() == expected_columns
assert len(train_original) == 9535 and train_original['Id'].is_unique
assert train_original['Answer'].apply(parse_answer).map(tuple).nunique() == 24

IDENTITY_ORDER = (1, 2, 3, 4)
source_answers = [tuple(parse_answer(value)) for value in train_original['Answer']]
sorted_indices = sorted(range(len(train_original)), key=lambda i: (source_answers[i], train_original.iloc[i]['Id']))
sorted_answers = [source_answers[i] for i in sorted_indices]
distribution_shift = int(max(pd.Series(source_answers).value_counts()))
shifted_answers = sorted_answers[distribution_shift:] + sorted_answers[:distribution_shift]
assert all(source != target for source, target in zip(sorted_answers, shifted_answers))
target_answer_by_index = [None] * len(train_original)
for position, row_index in enumerate(sorted_indices):
    target_answer_by_index[row_index] = shifted_answers[position]

def augment_row(row, target_answer):
    result = row.copy()
    old_inputs = [row[f'Input_{i}'] for i in IDENTITY_ORDER]
    old_answer = parse_answer(row['Answer'])
    target_answer = list(target_answer)
    order = tuple(old_answer.index(position) + 1 for position in target_answer)
    assert order != IDENTITY_ORDER
    for new_index, old_index in enumerate(order, start=1):
        result[f'Input_{new_index}'] = old_inputs[old_index - 1]
    result['Answer'] = str([old_answer[old_index - 1] for old_index in order])
    return result

augmented_train = pd.DataFrame([
    augment_row(row, target_answer_by_index[index])
    for index, (_, row) in enumerate(train_original.iterrows())
])
train_examples = pd.concat([train_original, augmented_train], ignore_index=True)
original_distribution = train_original['Answer'].apply(parse_answer).map(tuple).value_counts().sort_index()
augmented_distribution = augmented_train['Answer'].apply(parse_answer).map(tuple).value_counts().sort_index()
assert len(augmented_train) == 9535
assert len(train_examples) == 19070 and original_distribution.equals(augmented_distribution)
assert all(source != target for source, target in zip(
    train_original['Answer'].apply(parse_answer).map(tuple),
    augmented_train['Answer'].apply(parse_answer).map(tuple),
))
train_sha256 = hashlib.sha256(TRAIN_CSV.read_bytes()).hexdigest()
print(f'Train CSV SHA-256: {train_sha256}')
print(f'Full Train: {len(train_examples):,} = 원본 9,535 + 증강 9,535')
print('Validation/Test 접근: 없음')

### 4. 프롬프트·24개 class·6개 pair 관계 정의

프롬프트와 class 번호는 이전 모델과 동일합니다. 각 class가 여섯 이미지 쌍에서 어느 이미지가 먼저인지 나타내는 0/1 표를 추가로 만듭니다.

In [ ]:
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE,
)
STRUCTURED_HINT = (
    "This caption describes distinct sequential stages, often marked by words like 'then', "
    "'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    'This caption describes a single continuous action or a brief moment. '
    'Focus on subtle visual cues such as camera movement direction, object position changes, '
    'hand/body motion, and lighting changes to infer the temporal order.'
)

def split_clauses(sentence, max_clauses=4):
    parts = CLAUSE_PATTERN.split(sentence.strip())
    clauses = []
    current = parts[0].strip(' ,;.')
    for index in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[index + 1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [clause for clause in clauses if clause]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses - 1] + [' '.join(clauses[max_clauses - 1:])]
    return clauses or [sentence]

def answer_to_order(answer):
    answer = parse_answer(answer)
    return [index for _, index in sorted((position, index) for index, position in enumerate(answer, 1))]

def order_to_answer(order):
    answer = [0, 0, 0, 0]
    for position, image_number in enumerate(order, start=1):
        answer[image_number - 1] = position
    return answer

CHRONOLOGICAL_CLASSES = tuple(permutations(IDENTITY_ORDER))
CHRONOLOGICAL_TO_CLASS = {order: index for index, order in enumerate(CHRONOLOGICAL_CLASSES)}
IMAGE_PAIRS = tuple((first, second) for first in IDENTITY_ORDER for second in IDENTITY_ORDER if first < second)
assert len(CHRONOLOGICAL_CLASSES) == 24 and len(IMAGE_PAIRS) == 6

def answer_to_class_index(answer):
    return CHRONOLOGICAL_TO_CLASS[tuple(answer_to_order(answer))]

def class_index_to_answer(class_index):
    return order_to_answer(CHRONOLOGICAL_CLASSES[int(class_index)])

def class_pair_relations(order):
    positions = {image: position for position, image in enumerate(order)}
    return [float(positions[first] < positions[second]) for first, second in IMAGE_PAIRS]

PAIR_RELATION_MATRIX_CPU = torch.tensor(
    [class_pair_relations(order) for order in CHRONOLOGICAL_CLASSES], dtype=torch.float32
)
assert PAIR_RELATION_MATRIX_CPU.shape == (24, 6)
assert torch.all(PAIR_RELATION_MATRIX_CPU.sum(dim=0) == 12)
with SOURCE_MAPPING_PATH.open(encoding='utf-8') as file:
    stored_class_mapping = json.load(file)
expected_mapping = {str(index): list(order) for index, order in enumerate(CHRONOLOGICAL_CLASSES)}
assert stored_class_mapping['class_index_to_chronological_order'] == expected_mapping

def build_prompt_messages(row, image_dir):
    sentence = row['Sentence']
    hint = STRUCTURED_HINT if len(split_clauses(sentence)) >= 3 else CONTINUOUS_HINT
    content = []
    for image_number in IDENTITY_ORDER:
        image_path = image_dir / row['Id'] / row[f'Input_{image_number}']
        assert image_path.is_file(), image_path
        content.extend([
            {'type': 'image', 'image': str(image_path)},
            {'type': 'text', 'text': f'\nImage {image_number}\n'},
        ])
    content.append({'type': 'text', 'text': (
        f'Thinking about the sentence: \"{sentence}\"\n{hint}\n'
        'Look at the 4 images above labeled Image 1 to Image 4. '
        'Determine the correct chronological order of these images to match the sentence. '
        'Provide the answer ONLY as a Python list of integers. Example: [1, 2, 3, 4]'
    )})
    return [{'role': 'user', 'content': content}]

for answer in permutations(IDENTITY_ORDER):
    assert class_index_to_answer(answer_to_class_index(list(answer))) == list(answer)
print('24-class와 6개 pair 관계 정의 통과')

### 5. Dataset·Pairwise 학습 직전 모델 로드

`fold0_cls24_v1`의 LoRA와 classifier를 학습 가능한 상태로 불러옵니다. 이미 pairwise 학습된 adapter가 아니라 바로 이전 단계에서 시작해 기존 7,620개에 pairwise 학습이 중복되는 것을 막습니다.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
from safetensors.torch import load_file, save_file

class OrderingDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        return {
            'Id': row['Id'],
            'messages': build_prompt_messages(row, TRAIN_IMAGE_DIR),
            'Answer': row['Answer'],
            'class_label': answer_to_class_index(row['Answer']),
        }

@dataclass
class Collator:
    processor: object
    def __call__(self, examples):
        texts, all_images = [], []
        for example in examples:
            messages = example['messages']
            texts.append(self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            image_inputs, video_inputs = process_vision_info(messages)
            assert image_inputs is not None and len(image_inputs) == 4 and not video_inputs
            all_images.extend(image_inputs)
        batch = self.processor(text=texts, images=all_images, padding=True, return_tensors='pt')
        batch['class_labels'] = torch.tensor([e['class_label'] for e in examples], dtype=torch.long)
        batch['sample_ids'] = [e['Id'] for e in examples]
        batch['target_answers'] = [e['Answer'] for e in examples]
        return dict(batch)

processor = AutoProcessor.from_pretrained(MODEL_NAME)
processor.tokenizer.padding_side = 'right'
half_artifacts = [HALF_ADAPTER_DIR, HALF_CLASSIFIER_PATH, HALF_MAPPING_PATH]
final_artifacts = [FINAL_ADAPTER_DIR, CLASSIFIER_PATH, CLASS_MAPPING_PATH]
half_complete = all(path.exists() for path in half_artifacts)
final_complete = all(path.exists() for path in final_artifacts)
if any(path.exists() for path in half_artifacts) and not half_complete:
    raise RuntimeError(f'0.5 epoch 결과 폴더가 불완전합니다: {HALF_ADAPTER_DIR}')
if any(path.exists() for path in final_artifacts) and not final_complete:
    raise RuntimeError(f'결과 폴더가 불완전합니다: {FINAL_ADAPTER_DIR}')
if final_complete and not half_complete:
    raise RuntimeError(f'최종 결과는 있지만 0.5 epoch 결과가 없습니다: {HALF_ADAPTER_DIR}')
if half_complete and not final_complete:
    raise RuntimeError(
        '0.5 epoch까지만 저장된 중단 실행입니다. optimizer 상태를 저장하지 않았으므로 '
        'RUN_NAME을 바꿔 처음부터 다시 실행하세요.'
    )
adapter_to_load = FINAL_ADAPTER_DIR if final_complete else SOURCE_DIR
classifier_to_load = CLASSIFIER_PATH if final_complete else SOURCE_CLASSIFIER_PATH
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map='auto', low_cpu_mem_usage=True
)
model = PeftModel.from_pretrained(base_model, adapter_to_load, is_trainable=not final_complete)
loaded_lora = model.peft_config['default']
assert set(loaded_lora.target_modules) == {
    'q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'
}
assert loaded_lora.r == 16 and loaded_lora.lora_alpha == 32
model_device = next(model.parameters()).device
text_config = getattr(model.get_base_model().config, 'text_config', model.get_base_model().config)
HIDDEN_SIZE = int(text_config.hidden_size)
classifier = nn.Linear(HIDDEN_SIZE, 24, bias=True)
classifier.load_state_dict(load_file(str(classifier_to_load), device='cpu'))
classifier.to(model_device, dtype=torch.float32)
model.config.use_cache = False
if not final_complete:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.enable_input_require_grads()
multimodal_core = model.get_base_model().model
PAIR_RELATION_MATRIX = PAIR_RELATION_MATRIX_CPU.to(model_device)
training_dataset = OrderingDataset(train_examples)
collator = Collator(processor)

def move_batch(batch):
    labels = batch.pop('class_labels').to(model_device, non_blocking=True)
    sample_ids = batch.pop('sample_ids')
    target_answers = batch.pop('target_answers')
    inputs = {key: value.to(model_device, non_blocking=True) if torch.is_tensor(value) else value for key, value in batch.items()}
    return inputs, labels, sample_ids, target_answers

def pool_last_prompt_token(hidden, attention_mask):
    last_indices = attention_mask.sum(dim=1).long() - 1
    batch_indices = torch.arange(hidden.shape[0], device=hidden.device)
    return hidden[batch_indices, last_indices].float()

PAIR_EARLIER_MASK = PAIR_RELATION_MATRIX.bool().unsqueeze(0)

def class_logits_to_pair_logits(class_logits):
    expanded_logits = class_logits.unsqueeze(-1)
    earlier_logsumexp = torch.logsumexp(
        expanded_logits.masked_fill(~PAIR_EARLIER_MASK, float('-inf')), dim=1
    )
    later_logsumexp = torch.logsumexp(
        expanded_logits.masked_fill(PAIR_EARLIER_MASK, float('-inf')), dim=1
    )
    return earlier_logsumexp - later_logsumexp

audit = collator([training_dataset[0]])
audit_inputs, audit_labels, _, _ = move_batch(audit)
model.eval(); classifier.eval()
with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    audit_outputs = multimodal_core(**audit_inputs, use_cache=False, return_dict=True)
audit_hidden = pool_last_prompt_token(audit_outputs.last_hidden_state, audit_inputs['attention_mask'])
audit_logits = classifier(audit_hidden)
audit_pair_logits = class_logits_to_pair_logits(audit_logits)
audit_pair_targets = PAIR_RELATION_MATRIX[audit_labels]
audit_ce = F.cross_entropy(audit_logits, audit_labels)
audit_pair_loss = F.binary_cross_entropy_with_logits(audit_pair_logits, audit_pair_targets)
assert audit_logits.shape == (1, 24) and audit_pair_logits.shape == (1, 6)
assert torch.isfinite(audit_ce + audit_pair_loss)
print(f'시작 class CE: {audit_ce.item():.6f}')
print(f'초기 pairwise BCE: {audit_pair_loss.item():.6f}')
print(f'로드한 adapter: {adapter_to_load}')
print(f'Full Train examples: {len(training_dataset):,}')
print('기존 classifier 로드와 pairwise loss 감사 통과')
del audit, audit_inputs, audit_outputs, audit_hidden, audit_logits, audit_pair_logits
torch.cuda.empty_cache()

### 6. Train 전체 Pairwise 최종 단계 1 epoch

Fold 0에서 0.559791을 만든 설정을 변경하지 않고 재사용합니다: unweighted CE + `0.3 × pairwise BCE`, LoRA `2e-5`, classifier `1e-4`, effective batch 8입니다. 총 19,070개를 한 번 학습하며 약 0.5 epoch와 1 epoch 결과를 모두 저장합니다. 두 결과 중 선택은 별도 Test 추론·제출 단계에서 하며 이 노트북은 Test를 열지 않습니다.

In [ ]:
import bitsandbytes as bnb

BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LORA_LEARNING_RATE = 2e-5
CLASSIFIER_LEARNING_RATE = 1e-4
PAIRWISE_LOSS_WEIGHT = 0.3
DATALOADER_NUM_WORKERS = 2
LOG_EVERY_OPTIMIZER_STEPS = 10

run_config = {
    'scope': 'full_train_9535_no_validation_no_test_access',
    'train_sha256': train_sha256,
    'source_adapter': str(SOURCE_DIR),
    'raw_train_samples': len(train_original),
    'augmented_train_samples': len(augmented_train),
    'training_examples': len(training_dataset),
    'validation_samples': 0,
    'objective': 'unweighted_cross_entropy_plus_0.3_pairwise_binary_cross_entropy',
    'pairwise_implementation': 'vectorized_masked_logsumexp',
    'metric_sync_interval_optimizer_steps': LOG_EVERY_OPTIMIZER_STEPS,
    'image_pairs': [list(pair) for pair in IMAGE_PAIRS],
    'lora_learning_rate': LORA_LEARNING_RATE,
    'classifier_learning_rate': CLASSIFIER_LEARNING_RATE,
    'pairwise_loss_weight': PAIRWISE_LOSS_WEIGHT,
    'effective_batch_size': BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    'checkpoint_policy': 'half_epoch_and_full_epoch',
}
with (RUN_DIR / 'run_config.json').open('w', encoding='utf-8') as file:
    json.dump(run_config, file, ensure_ascii=False, indent=2)

def save_model_bundle(adapter_dir, classifier_path, mapping_path):
    adapter_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(adapter_dir, safe_serialization=True)
    processor.save_pretrained(adapter_dir)
    save_file(
        {
            key: value.detach().cpu().contiguous()
            for key, value in classifier.state_dict().items()
        },
        str(classifier_path),
    )
    with mapping_path.open('w', encoding='utf-8') as file:
        json.dump(stored_class_mapping, file, ensure_ascii=False, indent=2)

if not final_complete:
    torch.backends.cuda.matmul.allow_tf32 = True
    lora_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    classifier_parameters = list(classifier.parameters())
    optimizer = bnb.optim.AdamW8bit([
        {'params': lora_parameters, 'lr': LORA_LEARNING_RATE, 'weight_decay': 0.0},
        {'params': classifier_parameters, 'lr': CLASSIFIER_LEARNING_RATE, 'weight_decay': 0.01},
    ])
    optimizer.zero_grad(set_to_none=True)
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(
        training_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator,
        num_workers=DATALOADER_NUM_WORKERS, prefetch_factor=2, persistent_workers=True,
        pin_memory=True, generator=generator,
    )
    model.train(); classifier.train(); model.config.use_cache = False
    records, sample_count = [], 0
    correct_count_tensor = torch.zeros((), dtype=torch.long, device=model_device)
    ce_sum_tensor = torch.zeros((), dtype=torch.float32, device=model_device)
    pair_sum_tensor = torch.zeros((), dtype=torch.float32, device=model_device)
    total_sum_tensor = torch.zeros((), dtype=torch.float32, device=model_device)
    started = time.perf_counter()
    progress = tqdm(loader, desc='full_pair_train')
    optimizer_step = 0
    half_micro_step = (len(loader) // 2 // GRADIENT_ACCUMULATION_STEPS) * GRADIENT_ACCUMULATION_STEPS
    assert half_micro_step > 0 and half_micro_step % GRADIENT_ACCUMULATION_STEPS == 0
    half_saved = False
    for micro_step, batch in enumerate(progress, start=1):
        inputs, labels, _, _ = move_batch(batch)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            outputs = multimodal_core(**inputs, use_cache=False, return_dict=True)
        hidden = pool_last_prompt_token(outputs.last_hidden_state, inputs['attention_mask'])
        with torch.autocast(device_type='cuda', enabled=False):
            logits = classifier(hidden)
            ce_loss = F.cross_entropy(logits, labels)
            pair_logits = class_logits_to_pair_logits(logits)
            pair_targets = PAIR_RELATION_MATRIX[labels]
            pair_loss = F.binary_cross_entropy_with_logits(pair_logits, pair_targets)
            total_loss = ce_loss + PAIRWISE_LOSS_WEIGHT * pair_loss
        (total_loss / GRADIENT_ACCUMULATION_STEPS).backward()
        sample_count += labels.numel()
        correct_count_tensor += (logits.detach().argmax(dim=-1) == labels).sum()
        ce_sum_tensor += ce_loss.detach()
        pair_sum_tensor += pair_loss.detach()
        total_sum_tensor += total_loss.detach()
        should_update = micro_step % GRADIENT_ACCUMULATION_STEPS == 0 or micro_step == len(loader)
        if should_update:
            grad_norm = torch.nn.utils.clip_grad_norm_(
                lora_parameters + classifier_parameters, 1.0, error_if_nonfinite=True
            )
            optimizer.step(); optimizer.zero_grad(set_to_none=True)
            optimizer_step += 1
            should_log = (
                optimizer_step == 1
                or optimizer_step % LOG_EVERY_OPTIMIZER_STEPS == 0
                or micro_step == len(loader)
            )
            if should_log:
                ce_value = float(ce_loss.detach().item())
                pair_value = float(pair_loss.detach().item())
                total_value = float(total_loss.detach().item())
                running_correct = int(correct_count_tensor.item())
                running_accuracy = running_correct / sample_count
                records.append({
                    'micro_step': micro_step, 'optimizer_step': optimizer_step,
                    'ce_loss': ce_value, 'pairwise_loss': pair_value,
                    'total_loss': total_value, 'running_accuracy': running_accuracy,
                    'grad_norm': float(grad_norm),
                })
                progress.set_postfix(
                    ce=f'{ce_value:.4f}', pair=f'{pair_value:.4f}',
                    acc=f'{running_accuracy:.4f}', step=optimizer_step,
                )
            if micro_step == half_micro_step:
                save_model_bundle(
                    HALF_ADAPTER_DIR, HALF_CLASSIFIER_PATH, HALF_MAPPING_PATH
                )
                half_saved = True
                print(
                    f'약 0.5 epoch 저장: {HALF_ADAPTER_DIR} '
                    f'(micro step {micro_step:,}/{len(loader):,})'
                )
    assert half_saved
    elapsed_seconds = time.perf_counter() - started
    correct_count = int(correct_count_tensor.item())
    ce_sum = float(ce_sum_tensor.item())
    pair_sum = float(pair_sum_tensor.item())
    total_sum = float(total_sum_tensor.item())
    save_model_bundle(FINAL_ADAPTER_DIR, CLASSIFIER_PATH, CLASS_MAPPING_PATH)
    pd.DataFrame(records).to_csv(RUN_DIR / 'training_log.csv', index=False, encoding='utf-8-sig')
    training_summary = {
        'sample_count': sample_count, 'training_accuracy': correct_count / sample_count,
        'mean_ce_loss': ce_sum / len(loader), 'mean_pairwise_loss': pair_sum / len(loader),
        'mean_total_loss': total_sum / len(loader), 'elapsed_seconds': elapsed_seconds,
        'source_adapter': str(SOURCE_DIR),
        'half_epoch_adapter_dir': str(HALF_ADAPTER_DIR),
        'final_adapter_dir': str(FINAL_ADAPTER_DIR),
        'half_epoch_micro_step': half_micro_step,
        'objective': 'unweighted_cross_entropy_plus_0.3_pairwise_binary_cross_entropy',
        'validation_performed': False, 'test_accessed': False,
    }
    with (RUN_DIR / 'training_summary.json').open('w', encoding='utf-8') as file:
        json.dump(training_summary, file, ensure_ascii=False, indent=2)
    model.gradient_checkpointing_disable(); model.config.use_cache = True
    print(f'Full Train Pairwise 완료: {FINAL_ADAPTER_DIR}')
else:
    print(f'기존 완료 결과를 사용합니다: {FINAL_ADAPTER_DIR}')

### 7. Full Train 결과 파일 확인

전체 Train을 학습에 사용했기 때문에 이 노트북에서는 Validation 점수를 계산하거나 Test 데이터를 열지 않습니다. 0.5/1 epoch adapter와 학습 기록이 모두 저장됐는지만 확인합니다.

In [ ]:
required_artifacts = half_artifacts + final_artifacts + [
    RUN_DIR / 'run_config.json',
    RUN_DIR / 'training_log.csv',
    RUN_DIR / 'training_summary.json',
]
missing_artifacts = [path for path in required_artifacts if not path.exists()]
assert not missing_artifacts, f'저장되지 않은 결과가 있습니다: {missing_artifacts}'

with (RUN_DIR / 'training_summary.json').open(encoding='utf-8') as file:
    saved_summary = json.load(file)
training_log = pd.read_csv(RUN_DIR / 'training_log.csv')
assert saved_summary['sample_count'] == 19070
assert len(training_log) > 0
display(training_log.tail(10))
print(json.dumps(saved_summary, ensure_ascii=False, indent=2))
print(f'0.5 epoch adapter: {HALF_ADAPTER_DIR}')
print(f'1 epoch adapter: {FINAL_ADAPTER_DIR}')
print('다음 단계: 별도 Test 추론 노트북에서 두 adapter를 각각 추론합니다.')